# Data Restructuring & Saving

In this notebook, we will:
- Modify the **Spike Path DataFrame** using metadata dates.
- Add unique **hashes** for efficient storage and quick lookup of:
  - Sequence paths
  - Mutation paths

This restructuring will ensure:
- Consistent alignment of spike data with external metadata timelines.
- Fast and reliable retrieval of sequence/mutation information.
- A standardized format for downstream analysis and storage.

In [1]:
# Data handling
import pandas as pd
from datetime import datetime
import pickle
from tqdm import tqdm

# Sequence parsing
from Bio import SeqIO

# --- Custom imports ---
from Helpers import *   # Contains utility functions for data processing and plotting

In [2]:
# --- Load reference sequence and spike paths data ---

# Load reference spike sequence
ROOT_PATH = '/data/pgangwar/vaccinescout/data/'
refSeq = getReference(ROOT_PATH + "ref_spike_genome.fa")

# Load spike paths DataFrame
spike_paths_df = pd.read_parquet(ROOT_PATH +'all_nodes_spike_2025-09-01.parquet') 

# Quick sanity checks
print(f"Reference sequence length: {len(refSeq)} amino acids")
# Preview spike paths DataFrame
print(f"Loaded spike_paths_df with {spike_paths_df.shape[0]} rows and {spike_paths_df.shape[1]} columns")

Reference sequence length: 1274 amino acids
Loaded spike_paths_df with 21096844 rows and 2 columns


In [3]:
# --- Drop internal nodes ---
initial_rows = spike_paths_df.shape[0]
spike_paths_df = spike_paths_df[~spike_paths_df['sample_id'].str.contains('node_')]
dropped_rows = initial_rows - spike_paths_df.shape[0]
print(f"Dropped {dropped_rows} rows containing internal nodes in node_id")
print(f"Remaining spike_paths_df has {spike_paths_df.shape[0]} rows")

Dropped 3961557 rows containing internal nodes in node_id
Remaining spike_paths_df has 17135287 rows


In [4]:
# --- Extract dates from sample_id and report parsing failures ---

extracted_dates = []

for i, name in enumerate(spike_paths_df['sample_id']):
    parts = str(name).split('|')
    date_str = parts[-1].strip()

    try:
        # Attempt: Parse exactly as full date (YYYY-MM-DD)
        parsed_date = datetime.strptime(date_str, "%Y-%m-%d")
        extracted_dates.append(parsed_date)
    except ValueError:
        # If it fails, mark as invalid
        extracted_dates.append(pd.NaT)

# 1. Assign the extracted dates back to the dataframe
spike_paths_df['Date'] = extracted_dates

# 2. Count the invalid entries for reporting (without modifying the main df)
removed_count = spike_paths_df['Date'].isna().sum()
total_original = len(spike_paths_df)

# 3. Report the Results
print(f"Total original entries: {total_original}")
print(f"Invalid/incomplete dates to be removed: {removed_count}")

Total original entries: 17135287
Invalid/incomplete dates to be removed: 271802


In [5]:
# --- Sort spike_paths_df by Date ascending ---
# Filter out NaT, then sort, then reset the index all in one go
spike_paths_df = (
    spike_paths_df[spike_paths_df['Date'].notna()]
    .sort_values(by='Date', ascending=True)
    .reset_index(drop=True)
)

# --- Define date range to filter ---

start_date = pd.Timestamp('2020-01-01')
end_date = pd.Timestamp('2025-09-01')

# Filter rows to keep only those within the date range
spike_paths_df = spike_paths_df[
    (spike_paths_df['Date'] >= start_date) & (spike_paths_df['Date'] <= end_date)
].reset_index(drop=True)

print(f"Filtered spike_paths_df to {len(spike_paths_df)} entries between {start_date.date()} and {end_date.date()}")

Filtered spike_paths_df to 16863436 entries between 2020-01-01 and 2025-09-01


# Morphing Data: Creating Hashed Mutation and Sequence IDs

In this step, we will generate unique hash identifiers for:

- **Mutation sets**: to create compact Mutation IDs representing each unique mutation combination.
- **Sequences**: to create Sequence IDs that enable efficient storage and quick lookups.

This hashing significantly reduces data size and improves performance of downstream operations such as:

- Data merging and grouping.
- Fast mutation and sequence comparisons.
- Storage and retrieval optimizations.

The hash IDs serve as stable keys that abstract away verbose mutation or sequence details.

In [6]:
from tqdm import tqdm
import pandas as pd

tqdm.pandas()  # enable progress_apply / progress_map

def flatten_nested_array(val):
    """
    Converts standard mutation strings into a list format.
    """
    # 1. Check for missing data (NaN) or empty strings/spaces
    if pd.isna(val) or str(val).strip() == "":
        return []
    
    # 2. Return the string wrapped in a list. 
    # (Using .split(';') safely handles rows that might have multiple mutations like "S:P812L;S:A831V")
    return [m.strip() for m in str(val).split(';')]

# Apply with progress bar
spike_paths_df['aa_mutations'] = spike_paths_df['aa_mutations'].progress_map(flatten_nested_array)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 16863436/16863436 [01:45<00:00, 159255.49it/s]


In [7]:
from tqdm import tqdm
tqdm.pandas(desc="Applying path mutations")

def apply_path_mutations(path, refSeq):
    """
    Apply a list of mutations (path) to the reference sequence.

    Args:
        path (list): List of mutation strings.
        refSeq (str or Seq): Reference sequence.
    
    Returns:
        str: Mutated sequence string.
    """
    if not path:  # empty or None path
        return str(refSeq)
    return apply_mutations_flat_list(path, refSeq)

# Cache dictionary to avoid repeated computations
cache = {}

def mutate_row_cached(path):
    """
    Mutate a sequence for a given mutation path using caching for speed.

    Args:
        path (list): List of mutation strings representing mutation path.

    Returns:
        str: Mutated sequence string.
    """
    key = tuple(path)  # immutable key for caching
    if key not in cache:
        cache[key] = apply_path_mutations(path, refSeq)
    return cache[key]

# Apply mutation paths with progress bar, store results in new column
spike_paths_df['mutated_sequence'] = spike_paths_df['aa_mutations'].progress_apply(mutate_row_cached)

Applying path mutations: 100%|███████████████████████████████████████████████████████████████████| 16863436/16863436 [00:55<00:00, 302722.91it/s]


# Generating Consistent Mutation and Sequence Keys

To facilitate fast and reliable lookups, we create unique keys for each:

- **Mutation set**:  
  Each unique combination of mutations is converted into a consistent string, hashed, and prefixed with `M_` to form a compact Mutation Key.

- **Sequence**:  
  Each mutated amino acid sequence is hashed and prefixed with `S_` to generate a unique Sequence Key.

These keys provide:  
- Fixed-length identifiers for easy indexing and storage.  
- Consistency across datasets to avoid redundancy.  
- Improved efficiency for join and comparison operations.

In [8]:
# --- Generate Mutation_ID and Sequence_ID columns ---

# Convert parsed spike mutation lists to tuples for hashing and uniqueness
spike_paths_df['mutation_path_tuple'] = spike_paths_df['aa_mutations'].apply(tuple)

# Create a unique list of mutation paths and assign Mutation_IDs
unique_paths = spike_paths_df['mutation_path_tuple'].unique()
mutation_id_map = {path: f"M_{i}" for i, path in enumerate(unique_paths)}
spike_paths_df['Mutation_ID'] = spike_paths_df['mutation_path_tuple'].apply(lambda x: mutation_id_map[x])

# Create a unique list of mutated sequences and assign Sequence_IDs
unique_seqs = spike_paths_df['mutated_sequence'].unique()
sequence_id_map = {seq: f"S_{i}" for i, seq in enumerate(unique_seqs)}
spike_paths_df['Sequence_ID'] = spike_paths_df['mutated_sequence'].apply(lambda x: sequence_id_map[x])

In [9]:
# --- Create final_df and reverse lookup maps ---
final_df = spike_paths_df[['sample_id', 'Mutation_ID', 'Sequence_ID', 'Date']].copy()
final_df.rename(columns={'sample_id': 'Sample_ID'}, inplace=True)

# Reverse lookup dictionaries for sequences and mutation paths
sequence_dict = {v: k for k, v in sequence_id_map.items()}
mutation_path_dict = {v: k for k, v in mutation_id_map.items()}

In [10]:
# --- Preview of final_df ---
print(f"final_df shape: {final_df.shape}")
final_df.head()

final_df shape: (16863436, 4)


,Sample_ID,Mutation_ID,Sequence_ID,Date
0,TrinidadandTobago/3574/2020|EPI_ISL_16253523|2...,M_0,S_0,2020-01-01
1,Canada/BC-BCCDC-12035/2020|EPI_ISL_2522119|202...,M_1,S_1,2020-01-01
2,Wuhan/WH03/2020|LR757996.1|2020-01-01,M_2,S_2,2020-01-01
3,TrinidadandTobago/2208/2020|EPI_ISL_16253521|2...,M_3,S_3,2020-01-01
4,Wuhan/IVDC-HB-envF13-20/2020|NMDC60013073-01|2...,M_2,S_2,2020-01-01


In [11]:
import pandas as pd
from pathlib import Path

# --- Define output directory and paths ---
ROOT_PATH = Path('/data/pgangwar/vaccinescout/data/')
ROOT_PATH.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

final_df_path = ROOT_PATH / "final_df.parquet"
sequence_dict_path = ROOT_PATH / "sequence_dict.parquet"
mutation_path_dict_path = ROOT_PATH / "mutation_path_dict.parquet"

# --- Save final_df DataFrame as Parquet ---
final_df.to_parquet(final_df_path, compression="snappy", index=False)

# --- Save dictionaries as Parquet (Requires conversion to DataFrame first) ---

# Convert sequence_dict: creates a table with 'key' and 'value' columns
pd.DataFrame(
    list(sequence_dict.items()), 
    columns=['key', 'value']
).to_parquet(sequence_dict_path, compression="snappy", index=False)

# Convert mutation_path_dict: creates a table with 'key' and 'value' columns
pd.DataFrame(
    list(mutation_path_dict.items()), 
    columns=['key', 'value']
).to_parquet(mutation_path_dict_path, compression="snappy", index=False)

print(f"Saved final_df to {final_df_path}")
print(f"Saved sequence_dict to {sequence_dict_path}")
print(f"Saved mutation_path_dict to {mutation_path_dict_path}")

Saved final_df to /data/pgangwar/vaccinescout/data/final_df.parquet
Saved sequence_dict to /data/pgangwar/vaccinescout/data/sequence_dict.parquet
Saved mutation_path_dict to /data/pgangwar/vaccinescout/data/mutation_path_dict.parquet
